In [1]:
%pip install tensorflow

# Aqui importamos las libreriras

from midiutil import MIDIFile
import torch
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from transformers import AutoModelForMaskedLM
from IPython.display import Audio, display

In [ ]:
# Creamos una melodia simple

melodias_base = [[
        {"nombre": "C4", "pitch": 60, "inicio": 0.0, "duracion": 0.5, "volumen": 80},
        {"nombre": "D4", "pitch": 62, "inicio": 0.5, "duracion": 0.5, "volumen": 80},
        {"nombre": "E4", "pitch": 64, "inicio": 1.0, "duracion": 0.5, "volumen": 90},
        {"nombre": "G4", "pitch": 67, "inicio": 1.5, "duracion": 0.5, "volumen": 90},
        {"nombre": "E4", "pitch": 64, "inicio": 2.0, "duracion": 0.5, "volumen": 75},
        {"nombre": "D4", "pitch": 62, "inicio": 2.5, "duracion": 0.5, "volumen": 75},
        {"nombre": "C4", "pitch": 60, "inicio": 3.0, "duracion": 1.0, "volumen": 95}
    ],[
        {"nombre": "C4", "pitch": 60, "inicio": 0.0, "duracion": 0.5, "volumen": 85},
        {"nombre": "E4", "pitch": 64, "inicio": 0.5, "duracion": 0.5, "volumen": 85},
        {"nombre": "G4", "pitch": 67, "inicio": 1.0, "duracion": 0.5, "volumen": 95},
        {"nombre": "C5", "pitch": 72, "inicio": 1.5, "duracion": 0.5, "volumen": 95},
        {"nombre": "G4", "pitch": 67, "inicio": 2.0, "duracion": 0.5, "volumen": 80},
        {"nombre": "E4", "pitch": 64, "inicio": 2.5, "duracion": 0.5, "volumen": 80},
        {"nombre": "C4", "pitch": 60, "inicio": 3.0, "duracion": 1.0, "volumen": 90}
    ],[
        {"nombre": "A4", "pitch": 69, "inicio": 0.0, "duracion": 0.5, "volumen": 80},
        {"nombre": "G4", "pitch": 67, "inicio": 0.5, "duracion": 0.5, "volumen": 80},
        {"nombre": "E4", "pitch": 64, "inicio": 1.0, "duracion": 0.5, "volumen": 85},
        {"nombre": "D4", "pitch": 62, "inicio": 1.5, "duracion": 0.5, "volumen": 85},
        {"nombre": "C4", "pitch": 60, "inicio": 2.0, "duracion": 1.0, "volumen": 95}
    ]]

df_melodia = pd.DataFrame(melodias_base[0])
df_melodia

In [ ]:
# Funcion para escuchar una melodia

# Diccionario de frecuencias en Hz. para cada nota
frecuencias = {
    "C4": 261.63, "D4": 293.66, "E4": 329.63, "F4": 349.23, "G4": 392.00, "A4": 440.00, "B4": 493.88,
    "C5": 523.25
}

sample_rate = 44100

# Definimos una funcion para sintetizar audio desde una melodia
def melodia_a_audio(melodia, sample_rate=44100):
  duracion_total= max(
      nota["inicio"]+ nota["duracion"]
      for nota in melodia
  )

  audio = np.zeros(int(duracion_total*sample_rate))

  # Recorremos cada nota de la melodia
  for nota in melodia:

    frecuencia = frecuencias[nota["nombre"]]

    inicio_muestra = int(nota["inicio"]*sample_rate)

    duracion_muestras = int(nota["duracion"]*sample_rate)

    t = np.linspace(0,
                    nota["duracion"],
                    duracion_muestras,
                    endpoint=False)

    onda = np.sin(2*np.pi*frecuencia*t)

    onda = onda* (nota["volumen"]/127)

    fade_len= min(500, len(onda)//10)

    if fade_len > 0:
      onda[:fade_len]*= np.linspace(0,1,fade_len)
      onda[-fade_len:]*= np.linspace(1,0,fade_len)

    audio[inicio_muestra:inicio_muestra + duracion_muestras]+= onda

  max_audio = np.max(np.abs(audio))

  if max_audio > 0:
    audio = audio/ max_audio

  return audio

In [ ]:
# Escuchar el batch de melodias

for idx, melodia in enumerate(melodias_base):
  print(f"Melodia base {idx+1}")

  display(pd.DataFrame(melodia))

  audio_base = melodia_a_audio(melodia, sample_rate=sample_rate)

  display(Audio(audio_base, rate=sample_rate))

In [ ]:
# Exportamos la amelodia a un archivo MIDI
midi=MIDIFile(1)

track=0

time=0

tempo=120

midi.addTempo(track,time,tempo)

for nota in melodia:

  midi.addNote(
      track,
      channel=0,
      pitch=nota["pitch"],
      time=nota["inicio"],
      duration=nota["duracion"],
      volume=nota["volumen"]
  )


nombre_archivo_midi = "melodia_transformer.mid"
with open (nombre_archivo_midi, "wb") as archivo:
  midi.writeFile(archivo)

In [ ]:
# Representar la misma como tokens

def melodia_a_eventos(melodia):
  eventos_musicales=[]

  for nota in melodia:

    eventos_musicales.append(f"SET_VELOCITY_{nota["volumen"]}")

    eventos_musicales.append(f"NOTE_ON_{nota["nombre"]}")

    eventos_musicales.append(f"TIME_SHIFT_{nota["duracion"]}")

    eventos_musicales.append(f"NOTE_OFF_{nota["nombre"]}")

  return eventos_musicales

eventos_musicales = melodia_a_eventos(melodia)
print(eventos_musicales)

In [ ]:
# Corpus de eventos musicales

corpus_eventos = []

for melodia_base in melodias_base:

  eventos = melodia_a_eventos(melodia_base)

  corpus_eventos.extend(eventos)

corpus_eventos = corpus_eventos * 80

# Mostramos primeros eventos
print(corpus_eventos[:40])
print("\nTotal de eventos:", len(corpus_eventos))

In [ ]:
# Creamos el vocabulario musical

vocab_musical = {
    evento:indice
    for indice, evento in enumerate(sorted(set(corpus_eventos)))
}


id_to_event = {
    indice:evento
    for evento, indice in vocab_musical.items()
}

corpus_ids = np.array([vocab_musical[evento]
                        for evento in corpus_eventos])

vocab_size_music = len(vocab_musical)
print(vocab_musical)
print("Tamaño del vocabulario:", vocab_size_music)
print("Primeros ID's:", corpus_ids[:40])

In [ ]:
# Creamos secuencias de entrenamiento

# Numero de tokens que el modelo verá
window_size = 8

X_music=[]

y_music=[]

for i in range(len(corpus_ids)- window_size):

  X_music.append(corpus_ids[i:i+window_size])

  y_music.append(corpus_ids[i+window_size])

X_music = np.array(X_music)
y_music = np.array(y_music)

In [ ]:
# Definimos el bloque transformer

def transformer_causal_block(x, num_heads=2, key_dim=32, ff_dim=128, dropout_rate=0.1):
  attention_output = MultiHeadAttention(
      num_heads=num_heads,
      key_dim=key_dim
  )(x, x, use_causal_mask= True)

  attention_output = Dropout(dropout_rate)(attention_output)

  x= x+ attention_output

  x= LayerNormalization(epsilon=1e-6)(x)

  ff= Dense(ff_dim, activation="relu")(x)

  ff = Dense(x.shape[-1])(ff)

  ff = Dropout(dropout_rate)(ff)

  x= x+ ff

  x= LayerNormalization(epsilon=1e-6)(x)

  return x

In [ ]:
# Construimos el mini music transformer

embedding_dim = 64

inputs = Input(shape=(window_size,))

tokens_embeddings = Embedding(
    input_dim = vocab_size_music,
    output_dim = embedding_dim
)(inputs)

positions = tf.range(start=0, limit=window_size, delta=1)

position_embedding_layer= Embedding(
    input_dim = window_size,
    output_dim = embedding_dim
)

position_embeddings = position_embedding_layer(positions)

x = tokens_embeddings + position_embeddings

x= transformer_causal_block(x)

x= transformer_causal_block(x)

x = Lambda(lambda tensor: tensor[:, -1, :])(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.2)(x)

outputs = Dense(vocab_size_music, activation="softmax")(x)

modelo_music_tranformer = Model(inputs, outputs)

modelo_music_tranformer.compile(
    optimizer=Adam (learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_music_tranformer.summary()

In [ ]:
# Entrenamiento del modelo

hist_music= modelo_music_tranformer.fit(
    X_music,
    y_music,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Función de muestreo con temperatura

def sample_with_temperature(probabilidades, temperatura=1.0):

  probabilidades = np.asanyarray(probabilidades).astype("float64")
  probabilidades = np.log(probabilidades + 1e-9)/temperatura

  exp_probs = np.exp(probabilidades)
  probabilidades = exp_probs/np.sum(exp_probs)

  return np.random.choice(len(probabilidades), p=probabilidades)

In [ ]:
# Generamos nuevos tokens musicales

semilla = list(corpus_ids[:window_size])

secuencia_generada_ids= semilla.copy()

num_tokens_a_generar = 64

temperatura = 0.8

for _ in range(num_tokens_a_generar):

  contexto = secuencia_generada_ids[-window_size:]

  contexto = np.array([contexto])

  pred = modelo_music_tranformer.predict(contexto, verbose=0)[0]

  siguiente_id = sample_with_temperature(pred, temperatura=temperatura)
  secuencia_generada_ids.append(siguiente_id)

secuencia_generada_eventos = [
    id_to_event[i]
    for i in secuencia_generada_ids
]

In [ ]:
# Convertimos los eventos generados a melodía

nota_a_pitch = {
    "C4":60,
    "D4":62,
    "E4":64,
    "F4":65,
    "G4":67,
    "A4":69,
    "B4":71,
    "C5":72,
}

melodia_generada = []

tiempo_actual =0.0
volumen_actual=80
nota_actual = None
duracion_actual=0.5

for evento in secuencia_generada_eventos:

  if evento.startswith("SET_VELOCITY_"):
    volumen_actual = int(evento.replace("SET_VELOCITY_",""))
  elif evento.startswith("NOTE_ON_"):
    nota_actual = evento.replace("NOTE_ON_","")
  elif evento.startswith("TIME_SHIFT_"):
    duracion_actual = float(evento.replace("TIME_SHIFT_",""))
  elif evento.startswith("NOTE_OFF_"):
    if nota_actual in nota_a_pitch:
      melodia_generada.append({
          "nombre": nota_actual,
          "pitch": nota_a_pitch[nota_actual],
          "inicio": tiempo_actual,
          "duracion": duracion_actual,
          "volumen": volumen_actual
      })
      tiempo_actual += duracion_actual
    nota_actual=None

df_melodia_generada = pd.DataFrame(melodia_generada)
df_melodia_generada

In [ ]:
frecuencias = {
    "C4": 261.63, "D4": 293.66, "E4": 329.63, "F4": 349.23, "G4": 392.00, "A4": 440.00, "B4": 493.88,
    "C5": 523.25
}

sample_rate = 44100

if len(melodia_generada)==0:
  print("No se generaron notas validas. Prueba bajar la temperatura o entrenar mas epocas")
else:
  duracion_total= max(
      nota["inicio"]+ nota["duracion"]
      for nota in melodia_generada
  )

  audio = np.zeros(int(duracion_total*sample_rate))

  # Recorremos cada nota de la melodia
  for nota in melodia_generada:

    frecuencia = frecuencias[nota["nombre"]]

    inicio = int(nota["inicio"]*sample_rate)

    duracion = int(nota["duracion"]*sample_rate)

    t = np.linspace(0,
                    nota["duracion"],
                    duracion,
                    endpoint=False)

    onda = np.sin(2*np.pi*frecuencia*t)

    onda *= (nota["volumen"]/127)

    fade_len= min(500, len(onda)//10)
    onda[:fade_len]*= np.linspace(0,1,fade_len)
    onda[-fade_len:]*= np.linspace(1,0,fade_len)

    audio[inicio:inicio + duracion]+= onda
  audio = audio /np.max(np.abs(audio))
  display(Audio(audio, rate=sample_rate))